# Compare reconstructions — color

Evaluates reconstruction consistency across random instantiations of the reconstruction algorithm (e.g. STRND with different random seeds). Vertices are colored by their position in one reference reconstruction and the same colors are used to draw every other reconstruction. If the reconstructions agree, the color field appears smooth and continuous (up to rotation, reflection and scale) in all of them.

Images are saved to `$SPAN_DATA_ROOT/scratch/<project>/eval_color/`.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import time
from time import perf_counter
import random
import pickle
from collections import deque
from tqdm.auto import tqdm
from graph_tool import Graph
from graph_tool.draw import sfdp_layout, random_layout
from graph_tool.topology import label_components
import graph_tool as gt
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

print(gt.__file__)
print("Version: ", print(gt.__version__))
print("OpenMP Enabled: ", gt.openmp_enabled())

if gt.openmp_enabled():
    print("Number of threads before: ", gt.openmp_get_num_threads())
    gt.openmp_set_num_threads(1)
    print("Number of threads after: ", gt.openmp_get_num_threads())

## Helper functions

In [ ]:
# ----------------------------
# Helpers
# ----------------------------
def normalize_pos_95(pos):
    """
    pos: (N,2) float array
    Normalize so mean -> (0,0) and central 95% per-axis lies within [-1,1]
    using a single isotropic scale factor.
    Returns: (pos_norm, finite_mask)
    """
    pos = np.asarray(pos, dtype=np.float64)
    if pos.ndim != 2 or pos.shape[1] != 2:
        raise ValueError(f"pos must be Nx2; got {pos.shape}")

    finite = np.isfinite(pos).all(axis=1)
    pf = pos[finite]
    if pf.shape[0] == 0:
        raise ValueError("No finite positions to normalize.")

    # center
    mu = pf.mean(axis=0)
    p0 = pf - mu

    # scale (95% per-axis) with isotropic factor
    qlo = np.percentile(p0, 2.5, axis=0)
    qhi = np.percentile(p0, 97.5, axis=0)
    axis_halfspan = np.maximum(np.abs(qlo), np.abs(qhi))
    s = axis_halfspan.max()
    if not np.isfinite(s) or s == 0:
        raise ValueError("Degenerate layout: cannot determine nonzero scale.")

    pos_norm = np.full_like(pos, np.nan)
    pos_norm[finite] = p0 / s

    # --- Quick sanity checks (still NumPy-fast) ---
    mean_xy = np.nanmean(pos_norm, axis=0)
    qlo2 = np.nanpercentile(pos_norm, 2.5, axis=0)
    qhi2 = np.nanpercentile(pos_norm, 97.5, axis=0)

    print("mean (should be ~0,0):", mean_xy)
    print("2.5% quantiles:", qlo2, "  97.5% quantiles:", qhi2)
    print("max |quantile| per-axis:", np.maximum(np.abs(qlo2), np.abs(qhi2)))

    return pos_norm, finite


def left_vs_right_from_barcode(g, barcode_prop="barcode"):
    bc = g.vp[barcode_prop]
    lr = np.fromiter(
        (
            1 if (
                (x.decode("utf-8", "replace") if isinstance(x, (bytes, bytearray)) else str(x))
                .strip().removeprefix("b'").removesuffix("'")
                .startswith("L")
            ) else 0
            for x in (bc[v] for v in g.vertices())
        ),
        dtype=np.uint8,
        count=g.num_vertices(),
    )
    print("L count:", int(lr.sum()), "R/other count:", int(lr.size - lr.sum()))
    return lr


def colors_from_reference_positions(
    pos_ref_norm,
    min_saturation=0.40,
    max_saturation=0.90,
    brightness=0.68,
):
    """
    Assign one continuous 2D color to every vertex from its position in the
    reference layout.

    Mapping:
      - angle around the origin -> hue
      - distance from the origin -> saturation
      - brightness is held at a darker constant value

    Parameters
    ----------
    min_saturation : float
        Saturation near the center. Keeping this above zero prevents the
        center from becoming white or very pale.

    max_saturation : float
        Saturation near the outer part of the layout.

    brightness : float
        HSV value. Lower values produce darker colors.
        Suggested range: 0.55 to 0.75.
    """
    from matplotlib.colors import hsv_to_rgb

    pos_ref_norm = np.asarray(pos_ref_norm, dtype=np.float64)

    if pos_ref_norm.ndim != 2 or pos_ref_norm.shape[1] != 2:
        raise ValueError(
            f"pos_ref_norm must be Nx2; got {pos_ref_norm.shape}"
        )

    finite = np.isfinite(pos_ref_norm).all(axis=1)

    # Medium-dark gray for vertices without valid reference coordinates
    colors = np.full(
        (pos_ref_norm.shape[0], 4),
        [0.35, 0.35, 0.35, 1.0],
        dtype=np.float64,
    )

    x = pos_ref_norm[finite, 0]
    y = pos_ref_norm[finite, 1]

    # Position angle determines hue
    hue = (np.arctan2(y, x) + np.pi) / (2.0 * np.pi)

    # Position radius determines saturation
    radius = np.sqrt(x**2 + y**2)
    radius_scaled = np.clip(radius / np.sqrt(2.0), 0.0, 1.0)

    saturation = (
        min_saturation
        + radius_scaled * (max_saturation - min_saturation)
    )

    value = np.full_like(hue, brightness)

    hsv = np.column_stack([
        hue,
        saturation,
        value,
    ])

    colors[finite, :3] = hsv_to_rgb(hsv)

    return colors, finite


def plot_graph_with_reference_colors(
    pos_norm,
    vertex_colors,
    out_png,
    dpi=1000,
    marker_size=0.25,
    title=None,
    xlim=(-1.2, 1.2),
    ylim=(-1.2, 1.2),
):
    """
    Plot a layout using a precomputed per-vertex RGBA color array.

    vertex_colors should be generated once from the reference layout and then
    passed unchanged for the reference and comparison layouts.
    """
    import os

    pos_norm = np.asarray(pos_norm, dtype=np.float64)
    vertex_colors = np.asarray(vertex_colors, dtype=np.float64)

    if pos_norm.ndim != 2 or pos_norm.shape[1] != 2:
        raise ValueError(f"pos_norm must be Nx2; got {pos_norm.shape}")
    if vertex_colors.shape != (pos_norm.shape[0], 4):
        raise ValueError(
            f"vertex_colors must have shape {(pos_norm.shape[0], 4)}; "
            f"got {vertex_colors.shape}"
        )

    finite = np.isfinite(pos_norm).all(axis=1)

    os.makedirs(os.path.dirname(out_png), exist_ok=True)

    plt.figure(figsize=(6, 6))
    if title:
        plt.title(title)

    plt.scatter(
        pos_norm[finite, 0],
        pos_norm[finite, 1],
        s=marker_size,
        marker='o',
        c=vertex_colors[finite],
        linewidths=0,
        rasterized=True,
    )

    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xticks([])
    plt.yticks([])
    plt.box(False)

    plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("Saved:", out_png, "| total:", len(pos_norm), "finite:", int(finite.sum()))


def plot_2d_color_key(
    out_png,
    dpi=500,
    resolution=600,
    min_saturation=0.40,
    max_saturation=0.90,
    brightness=0.68,
):
    """
    Save a key showing the same darker continuous 2D position-to-color mapping.
    """
    import os
    from matplotlib.colors import hsv_to_rgb

    axis = np.linspace(-1.0, 1.0, resolution)
    xx, yy = np.meshgrid(axis, axis)

    hue = (np.arctan2(yy, xx) + np.pi) / (2.0 * np.pi)

    radius = np.sqrt(xx**2 + yy**2)
    radius_scaled = np.clip(radius / np.sqrt(2.0), 0.0, 1.0)

    saturation = (
        min_saturation
        + radius_scaled * (max_saturation - min_saturation)
    )

    value = np.full_like(hue, brightness)

    rgb = hsv_to_rgb(
        np.dstack([
            hue,
            saturation,
            value,
        ])
    )

    os.makedirs(os.path.dirname(out_png), exist_ok=True)

    plt.figure(figsize=(6, 6))
    plt.imshow(
        rgb,
        origin="lower",
        extent=(-1, 1, -1, 1),
        interpolation="bilinear",
    )

    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel("normalized reference x")
    plt.ylabel("normalized reference y")

    plt.savefig(
        out_png,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close()

    print("Saved:", out_png)


def load_pos_from_csv(csv_path):
    df_pos = pd.read_csv(csv_path)

    # --- build positions from dataframe, aligned to original graph node IDs ---
    # Expect columns: node_ID, x, y
    required = {"node_ID", "x", "y"}
    missing = required - set(df_pos.columns)
    if missing:
        raise ValueError(f"CSV missing columns {sorted(missing)}. Found: {list(df_pos.columns)}")

    # Sort by node_ID and construct (N,2) array
    df_pos = df_pos.sort_values("node_ID", kind="mergesort").reset_index(drop=True)

    node_ids = df_pos["node_ID"].to_numpy()
    if node_ids.size == 0:
        raise ValueError("Positions CSV is empty.")

    # Basic checks: integer-ish, unique, contiguous
    if not np.issubdtype(node_ids.dtype, np.integer):
        node_ids = pd.to_numeric(df_pos["node_ID"], errors="raise").astype(np.int64).to_numpy()

    if len(np.unique(node_ids)) != len(node_ids):
        raise ValueError("node_ID column contains duplicates.")

    expected = np.arange(node_ids.min(), node_ids.min() + len(node_ids), dtype=node_ids.dtype)
    if not np.array_equal(node_ids, expected):
        raise ValueError(
            "node_IDs are not contiguous after sorting. "
            f"min={node_ids.min()}, max={node_ids.max()}, count={len(node_ids)}"
        )

    pos = df_pos[["x", "y"]].to_numpy(dtype=np.float64, copy=False)
    return pos


## Parameters and graph
The graph is only used to check the vertex count.

In [ ]:
# Parameters
project_name = "260921Jac"
sample_name = "30_B5"

# Suffix of the graph that was reconstructed in pipeline step 07
suffix = "postPipeline0302_Lev3_min_e4_min_d2_max_d30_LCC"

data_dir = os.path.join(os.environ.get("SPAN_DATA_ROOT", "./orcd"), "scratch", project_name)

filename = f"{data_dir}/{sample_name}_barcodesGraph_{suffix}.gt.gz"

print("Loading: ", filename)

g = gt.load_graph(filename)

In [ ]:
num_nodes = g.num_vertices()
num_edges = g.num_edges()

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of edges: {num_edges:,}")

In [ ]:
left_or_right = left_vs_right_from_barcode(g)

#left_or_right = np.random.choice([0, 1], size=g.num_vertices(), p=[0.333, 0.667])

## Reconstructions to compare

In [ ]:
# Reconstructions to compare, by the tag in their step-07 file names
# (<SAMPLE>_positions_<SUFFIX>_STRND_<tag>.csv, tag = d<node_embedding_components>_rnd<random_state>).
# The first one is the reference.
strnd_layouts_to_compare = [
    "d32_rnd42",
    "d32_rnd999", 
    "d32_rnd123"
]

## Color by reference position and paint the others
Each vertex gets a color from its position in the reference reconstruction (angle → hue, distance from the center → saturation). The same per-vertex colors are then used to draw every other reconstruction, and a color key is saved.

In [ ]:
# Choose one layout as the reference color coordinate system.
# Every vertex receives its color from this layout only.
reference_layout = strnd_layouts_to_compare[0]
comparison_layouts = [x for x in strnd_layouts_to_compare if x != reference_layout]

# Output settings
dpi = 500
marker_size = 0.2
output_dir = f"{data_dir}/eval_color"

print(f"Using STRND_{reference_layout} as the reference color layout")

# Load and normalize the reference coordinates
reference_csv_path = (
    f"{data_dir}/{sample_name}_positions_{suffix}_STRND_{reference_layout}.csv"
)
print("Loading:", reference_csv_path)
reference_pos = load_pos_from_csv(reference_csv_path)
reference_pos_norm, reference_finite = normalize_pos_95(reference_pos)

if reference_pos_norm.shape[0] != g.num_vertices():
    raise ValueError(
        f"Reference CSV has {reference_pos_norm.shape[0]:,} rows, "
        f"but graph has {g.num_vertices():,} vertices."
    )

# Color appearance settings
min_saturation = 0.40
max_saturation = 0.90
color_brightness = 0.68

vertex_colors, color_finite = colors_from_reference_positions(
    reference_pos_norm,
    min_saturation=min_saturation,
    max_saturation=max_saturation,
    brightness=color_brightness,
)

plot_2d_color_key(
    out_png=f"{output_dir}/{sample_name}_{suffix}_ref{reference_layout}_color_key.png",
    min_saturation=min_saturation,
    max_saturation=max_saturation,
    brightness=color_brightness,
)

# Plot the reference layout
plot_graph_with_reference_colors(
    pos_norm=reference_pos_norm,
    vertex_colors=vertex_colors,
    out_png=f"{output_dir}/{sample_name}_{suffix}_ref{reference_layout}.png",
    dpi=dpi,
    marker_size=marker_size,
    title=None,
)

# Plot every other layout with the exact same per-vertex colors
for comparison_layout in comparison_layouts:
    print()
    print(f"Plotting STRND_{comparison_layout} with colors from STRND_{reference_layout}")

    comparison_csv_path = (
        f"{data_dir}/{sample_name}_positions_{suffix}_STRND_{comparison_layout}.csv"
    )
    print("Loading:", comparison_csv_path)

    comparison_pos = load_pos_from_csv(comparison_csv_path)
    comparison_pos_norm, comparison_finite = normalize_pos_95(comparison_pos)

    if comparison_pos_norm.shape[0] != reference_pos_norm.shape[0]:
        raise ValueError(
            f"Comparison CSV STRND_{comparison_layout} has "
            f"{comparison_pos_norm.shape[0]:,} rows, but the reference has "
            f"{reference_pos_norm.shape[0]:,}."
        )

    plot_graph_with_reference_colors(
        pos_norm=comparison_pos_norm,
        vertex_colors=vertex_colors,
        out_png=(
            f"{output_dir}/{sample_name}_{suffix}_"
            f"ref{reference_layout}_comp{comparison_layout}.png"
        ),
        dpi=dpi,
        marker_size=marker_size,
        title=None,
    )
